In [1]:
import sys
import os
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

sys.path.insert(0, "../src")

from extract_valid_studies import main
from models import ValidStudy, PrimaryOutcome

load_dotenv()

True

# Clinical Trial Card Generation - Batch API Version

This notebook uses OpenAI's **Batch API** for processing clinical trials at scale.

## Batch API Benefits
- 💰 **50% cost savings** compared to standard API
- 📦 Process up to **50,000 requests** per batch
- ⏱️ Results within **24 hours**
- 🔄 **Asynchronous processing** - submit and check later

## Workflow
1. **Prepare**: Create batch requests from studies
2. **Submit**: Upload and submit batch job
3. **Wait**: Batch processes asynchronously (up to 24h)
4. **Check**: Monitor batch status
5. **Download**: Retrieve and parse results when complete

## Important Notes
- Save your **batch ID** - you'll need it to check status and download results
- Batches process asynchronously - you can close this notebook after submission
- Check status periodically or use the monitoring function
- Results are available for download once status is "completed"

In [2]:
MODEL = "gpt-5-mini"
RAW_STUDIES_DIR = "../data/raw_studies"
STUDIES_PROCESSED_OUTPUT_DIR= "../data/studies_processed"
CARDS_OUTPUT_DIR = "../data/cards"
BATCH_INPUT_FILE = "../data/batch_input.jsonl"
BATCH_OUTPUT_FILE = "../data/batch_output.jsonl"
NUM_STUDIES = 99999

In [3]:
from pydantic import BaseModel, Field, model_validator

class LLMResponse(BaseModel):
    question: str = Field(..., description="The final question in the specified format with the appropriate placeholders filled in verbatim with the other fields.")
    intervention_fragment: str = Field(..., description="The main intervention being tested, in layperson's terms. This should be directly pluggable into the question template.")
    intervention_group_fragment: str = Field(..., description="The purpose of the clinical trial, in layperson's terms. This should be directly pluggable into the question template.")
    outcome_fragment: str = Field(..., description="The primary outcome being measured, in layperson's terms. This should be directly pluggable into the question template.")
    comparator_group_fragment: str = Field(..., description="The comparator or control condition, in layperson's terms. This should be directly pluggable into the question template.")
    timeframe_fragment: str = Field(..., description="The timeframe of the outcome measurement, in layperson's terms. This should be directly pluggable into the question template.")
    intervention_group_description: str = Field(..., description="A brief description of the intervention group.")
    comparator_group_description: str = Field(..., description="A brief description of the comparator/control group.")
    understandability_score: int = Field(..., description="A 1-10 score representing the understandability of the question by a lay reader.")

    @model_validator(mode="after")
    def ensure_question_valid(self):
        expected_question = f"Did {self.intervention_fragment} improve {self.outcome_fragment} in {self.intervention_group_fragment} compared to {self.comparator_group_fragment} after {self.timeframe_fragment}?"
        if self.question != expected_question:
            raise ValueError(f"Question does not match the expected format. Got: {self.question}, Expected: {expected_question}")
        return self


@dataclass
class ProcessingInformation:
    study: ValidStudy
    outcome_id: str
    llm_response: LLMResponse

    def to_dict(self):
        def recursive_asdict(obj):
            if isinstance(obj, list):
                return [recursive_asdict(item) for item in obj]
            elif isinstance(obj, dict):
                return {key: recursive_asdict(value) for key, value in obj.items()}
            elif hasattr(obj, "__dict__"):
                return {key: recursive_asdict(value) for key, value in obj.__dict__.items()}
            else:
                return obj
        return {
            "study": recursive_asdict(self.study),
            "outcome_id": self.outcome_id,
            "llm_response": self.llm_response.model_dump(),
        }

In [4]:
@dataclass
class UsageTracker:
    total_api_calls: int = 0
    total_input_tokens: int = 0
    total_output_tokens: int = 0


    def cost(self, use_batch_pricing: bool = True) -> float:
        """
        Calculate estimated cost.
        
        Args:
            use_batch_pricing: If True, apply 50% discount for Batch API
        """
        # Standard pricing per 1M tokens
        c_i = {
            "gpt-5": 1.25,
            "gpt-5-mini": 0.25,
        }
        c_o = {
            "gpt-5": 10,
            "gpt-5-mini": 2,
        }
        
        input_cost = (self.total_input_tokens / 1_000_000) * c_i[MODEL]
        output_cost = (self.total_output_tokens / 1_000_000) * c_o[MODEL]
        total_cost = input_cost + output_cost
        
        # Batch API gives 50% discount
        if use_batch_pricing:
            total_cost *= 0.5

        return total_cost

    def summary(self, use_batch_pricing: bool = True):
        print(f"Total API calls: {self.total_api_calls}")
        print(f"Total input tokens: {self.total_input_tokens:,}")
        print(f"Total output tokens: {self.total_output_tokens:,}")
        
        if use_batch_pricing:
            print(f"Estimated cost (Batch API with 50% discount): ${self.cost(use_batch_pricing=True):.4f}")
            print(f"  (Standard API would cost: ${self.cost(use_batch_pricing=False):.4f})")
        else:
            print(f"Estimated cost: ${self.cost(use_batch_pricing=False):.4f}")

tracker = UsageTracker()

In [5]:
def mk_prompt(v: ValidStudy, o: PrimaryOutcome) -> str:
    groups_info = []
    for g in o.groups:
        groups_info.append("\n".join([
            f"Group title: {g.title}",
            f"Description: {g.description}",
            f"Interventions: {', '.join([f"{i.name}: {i.description}" for i in g.interventions]) if g.interventions else '(uncertain)'}",
        ]))


    return f"""
We are creating flashcard summaries for a game where laypeople predict the outcomes of clinical trials (behavioral interventive).

The final question for the flashcard must be of format:
"Did [intervention_fragment] improve [outcome_fragment] in [intervention_group_fragment] compared to [comparator_group_fragment] after [timeframe_fragment]?"

Keep the questions as short as possible. Use acronyms if needed, as long as they are understandable. If there is a name given to the intervention (e.g. "The Jolly Flower Telephone Protocol for Healthy Ageing"), instead of using the name, simply describe the intervention in layperson's terms (e.g. "calling other elderly people").

Remember to keep the question short. Do not include examples. Do not include any additional text or explanation.

Recall that the question should be reconstructable by verbatim plugging in the other fields.


Please ensure that the answers are concise and easily understandable by someone without a medical background. Avoid technical jargon and use simple language. Where something is technical, give a lay description and then in parentheses the technical term. 

Please create a question based on the following clinical trial information:
• Trial Title: {v.title}
• Trial Description: {v.description}
• Measure: {o.title}
• Measure Description: {o.description}
• Timeframe: {o.timeframe}

The groups are as follows (the first is the intervention group):
{'\n\n'.join(groups_info)}


If there is missing intervention or comparator information, please either match to these interventions (if you can tell from the group title/description), or say "Control" if it is a no-treatment or standard care control group, or "Unknown" if you cannot tell.


Here are negative examples of questions:

EXAMPLE 1
Did using a web-based lung cancer screening decision aid improve decisional conflict (uncertainty about the screening decision) in Veterans who used the decision tool compared to Veterans given general prevention info (not about lung cancer) after 1 month?

Understandability score: 7/10 (confusing phrasing)

Better version:
Did using a web-based lung cancer screening decision aid reduce uncertainty in deciding whether to screen in Veterans who used the decision tool compared to Veterans given general prevention info (not about lung cancer) after 1 month?

Understandability score: 9/10


EXAMPLE 2
Did upregulating the left amygdala with real-time fMRI neurofeedback (thinking of positive memories) improve depressive symptoms (MADRS score) in MDD patients receiving left amygdala neurofeedback compared to MDD patients receiving HIPS (non-emotional region) neurofeedback after 2 weeks?

Understandability score: 3/10 (unclear what MDD is, what HIPS is)

Problems:
• Intervention seems to be "thinking of positive memories", the rest seems extraneous)
• MDD is not explained


EXAMPLE 3
Did wearing a UV dosimeter sticker and receiving daily personalized text messages based on sensor readings improve acceptability of wearing the UV sensor and receiving texts (system usability score, 6–42; higher better) in melanoma survivors in Cohort Study 1 Arm 1 (n=31) compared to melanoma survivors in Cohort Study 1 Arm 2 (n=29; daily texts + unstructured goal responses) after 21 days?


Understandability score: 5/10 (what is a dosimeter?)

Better version:
Did wearing a UV sensor sticker (UV dosimeter) and receiving daily personalized text messages improve acceptability of wearing a UV sensor and receiving related texts in melanoma survivors compared to melanoma survivors receiving daily texts + unstructured goal responses after 21 days?


EXAMPLE 4
Did family- and home-based behavioral support for ADHD (CASH‑AA) improve ADHD symptoms and related problems (delinquency, substance use, internalizing/externalizing symptoms) in adolescents with ADHD in the behavioral-only group compared to adolescents receiving the behavioral program plus medication integration (MIP) after one year?

Understandability score: 9/10

Better version (less non-core details):
Did family- and home-based behavioral support for ADHD (CASH‑AA) improve ADHD symptoms and related problems in adolescents with ADHD in the behavioral-only group compared to adolescents receiving the behavioral program plus medication integration (MIP) after one year?

Understandability score: 9/10 (no improvement, but more terse)


EXAMPLE 5
Did onsite collaborative care with a care manager (CC) improve treatment engagement (completed baseline and >2 OUD treatment visits) in pregnant and postpartum women with opioid use disorder compared to remote ECHO video mentorship (tele-support for providers) after 30 days from baseline?

Understandability score: 4/10 (what is ECHO, OUD?)

Problems:
• OUD is not explained
• ECHO is not explained


EXAMPLE 6
Did using an AF decision-support tool to recommend antithrombotic therapy improve discordant antithrombotic therapy (patients on treatment that disagreed with the tool's recommendation) in adults with non-valvular AF in primary care compared to educational intervention only (educational conference series) after one year?

Understandability score: 3/10 (what is AF?)

Problems:
• AF is not explained
• Antithrombotic is an unexplained technical term
• The explanation of "patients on treatment that disagreed with the tool's recommendation" is wordy and confusing.

---------------


Here are positive examples of questions:
EXAMPLE 1
Did practicing Tai Chi improve knee pain (WOMAC pain score) in people with knee osteoarthritis compared to wellness education and stretching after 12 weeks?

Understandability score: 10/10

Reason:
Concise, easy to understand, technical term in parenthesis (understanding is not gated by needing to know the technical term).


EXAMPLE 2
Did abstinence-only sex education improve abstinence (not having sex) in young African-American adolescents compared to health-promotion control after 24 months?

Understandability score: 10/10

Reason:
Easy to understand, clear treatment and control groups, short.


EXAMPLE 3
Did video counseling for quitting smoking improve smoking abstinence (verified by salivary cotinine) in women with HIV compared to women with HIV with telephone counseling instead after 6 months?

Understandability score: 10/10

Reason:
Good details, clear
    """


In [6]:
valid_studies = main(RAW_STUDIES_DIR)

100%|██████████| 34562/34562 [00:09<00:00, 3763.82it/s]


Loaded 5858 raw studies with results
2033 out of 5858 (34.70%) studies have p-values reported in primary outcomes analyses.


In [7]:
import json
import time
from pathlib import Path

def create_batch_requests(studies: list[ValidStudy]) -> list[dict]:
    """Create batch API requests for all study outcomes."""
    batch_requests = []
    request_index = 0  # Add a unique index to ensure no duplicates
    
    for study in studies:
        for outcome in study.primary_outcomes:
            # Create a unique custom_id for each request using index
            custom_id = f"{study.nct_id}_{outcome.id}_{request_index}"
            request_index += 1
            
            # Create the request in the format required by Batch API
            request = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are an expert clinical trial analyst."
                        },
                        {
                            "role": "user",
                            "content": mk_prompt(study, outcome)
                        }
                    ],
                    "response_format": {
                        "type": "json_schema",
                        "json_schema": {
                            "name": "LLMResponse",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "question": {"type": "string"},
                                    "intervention_fragment": {"type": "string"},
                                    "intervention_group_fragment": {"type": "string"},
                                    "outcome_fragment": {"type": "string"},
                                    "comparator_group_fragment": {"type": "string"},
                                    "timeframe_fragment": {"type": "string"},
                                    "intervention_group_description": {"type": "string"},
                                    "comparator_group_description": {"type": "string"},
                                    "understandability_score": {"type": "integer"}
                                },
                                "required": [
                                    "question",
                                    "intervention_fragment",
                                    "intervention_group_fragment",
                                    "outcome_fragment",
                                    "comparator_group_fragment",
                                    "timeframe_fragment",
                                    "intervention_group_description",
                                    "comparator_group_description",
                                    "understandability_score"
                                ],
                                "additionalProperties": False
                            }
                        }
                    }
                }
            }
            batch_requests.append(request)
    
    return batch_requests


def save_batch_input_file(batch_requests: list[dict], filepath: str):
    """Save batch requests to JSONL file."""
    with open(filepath, 'w') as f:
        for request in batch_requests:
            f.write(json.dumps(request) + '\n')
    print(f"Saved {len(batch_requests)} requests to {filepath}")


def submit_batch_job(client: OpenAI, input_file_path: str) -> str:
    """Upload the batch input file and create a batch job."""
    # Upload the file
    print(f"Uploading batch input file: {input_file_path}")
    with open(input_file_path, 'rb') as f:
        batch_input_file = client.files.create(
            file=f,
            purpose="batch"
        )
    
    print(f"File uploaded with ID: {batch_input_file.id}")
    
    # Create the batch job
    batch_job = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
            "description": "Clinical trial question generation"
        }
    )
    
    print(f"Batch job created with ID: {batch_job.id}")
    print(f"Status: {batch_job.status}")
    
    return batch_job.id


def check_batch_status(client: OpenAI, batch_id: str):
    """Check the status of a batch job."""
    batch = client.batches.retrieve(batch_id)
    print(f"\nBatch ID: {batch.id}")
    print(f"Status: {batch.status}")
    print(f"Total requests: {batch.request_counts.total}")
    print(f"Completed: {batch.request_counts.completed}")
    print(f"Failed: {batch.request_counts.failed}")
    
    if batch.status == "completed":
        print(f"Output file ID: {batch.output_file_id}")
    elif batch.status == "failed":
        print(f"Error file ID: {batch.error_file_id}")
    
    return batch


def download_batch_results(client: OpenAI, batch_id: str, output_path: str):
    """Download and save batch results."""
    batch = client.batches.retrieve(batch_id)
    
    if batch.status != "completed":
        print(f"Batch not completed yet. Status: {batch.status}")
        return None
    
    # Download the output file
    file_response = client.files.content(batch.output_file_id)
    
    # Save to file
    with open(output_path, 'wb') as f:
        f.write(file_response.content)
    
    print(f"Results saved to {output_path}")
    return output_path


def parse_batch_results(output_file_path: str, studies: list[ValidStudy]) -> tuple[list[ProcessingInformation], list[tuple[str, str]]]:
    """Parse the batch output file and create ProcessingInformation objects."""
    results = []
    failures = []
    
    # Create lookup for studies by custom_id (with index)
    study_outcome_map = {}
    request_index = 0
    for study in studies:
        for outcome in study.primary_outcomes:
            custom_id = f"{study.nct_id}_{outcome.id}_{request_index}"
            study_outcome_map[custom_id] = (study, outcome)
            request_index += 1
    
    # Read and parse the JSONL output
    with open(output_file_path, 'r') as f:
        for line in f:
            result = json.loads(line)
            custom_id = result['custom_id']
            
            if result.get('response'):
                try:
                    # Extract the response
                    response_body = result['response']['body']
                    content = response_body['choices'][0]['message']['content']
                    
                    # Parse the JSON response
                    llm_response_dict = json.loads(content)
                    llm_response = LLMResponse(**llm_response_dict)
                    
                    # Get the corresponding study and outcome
                    study, outcome = study_outcome_map[custom_id]
                    
                    # Create ProcessingInformation
                    processing_info = ProcessingInformation(
                        study=study,
                        outcome_id=outcome.id,
                        llm_response=llm_response
                    )
                    results.append(processing_info)
                    
                    # Track token usage
                    usage = response_body.get('usage', {})
                    tracker.total_api_calls += 1
                    tracker.total_input_tokens += usage.get('prompt_tokens', 0)
                    tracker.total_output_tokens += usage.get('completion_tokens', 0)
                    
                except Exception as e:
                    nct_id = custom_id.split('_')[0]
                    failures.append((nct_id, f"Parse error: {str(e)}"))
            else:
                # Handle error in the batch result
                nct_id = custom_id.split('_')[0]
                error_msg = result.get('error', {}).get('message', 'Unknown error')
                failures.append((nct_id, error_msg))
    
    return results, failures

In [8]:
# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Step 1: Create batch requests
print("Creating batch requests...")
batch_requests = create_batch_requests(valid_studies[:NUM_STUDIES])
print(f"Created {len(batch_requests)} batch requests")

# Step 2: Save to JSONL file
save_batch_input_file(batch_requests, BATCH_INPUT_FILE)

# Step 3: Submit batch job
batch_id = submit_batch_job(client, BATCH_INPUT_FILE)
print(f"\nBatch submitted! Batch ID: {batch_id}")
print("Save this batch ID to check status later!")

Creating batch requests...
Created 7336 batch requests
Saved 7336 requests to ../data/batch_input.jsonl
Uploading batch input file: ../data/batch_input.jsonl
File uploaded with ID: file-Kxzo9qM6KCEKMpE9QWCYuM
Batch job created with ID: batch_68ea113f29ac8190810b220833f280ff
Status: validating

Batch submitted! Batch ID: batch_68ea113f29ac8190810b220833f280ff
Save this batch ID to check status later!


In [9]:
# Optional: List all your previous batch jobs
def list_batch_jobs(client: OpenAI, limit: int = 10):
    """List recent batch jobs."""
    batches = client.batches.list(limit=limit)
    
    print(f"Recent batch jobs (showing up to {limit}):\n")
    for batch in batches.data:
        print(f"ID: {batch.id}")
        print(f"  Status: {batch.status}")
        print(f"  Created: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(batch.created_at))}")
        print(f"  Requests: {batch.request_counts.completed}/{batch.request_counts.total} completed")
        if batch.metadata:
            print(f"  Description: {batch.metadata.get('description', 'N/A')}")
        print()

# Uncomment to list your batches:
# list_batch_jobs(client)

## Check Batch Status

Run this cell to check the status of your batch job. Replace `BATCH_ID` with the ID from the previous cell.

The batch will be in one of these states:
- `validating` - checking the input file
- `in_progress` - processing requests
- `finalizing` - writing results
- `completed` - finished successfully
- `failed` - encountered an error
- `cancelled` - manually cancelled

In [10]:
# Check status of batch job
# Replace with your actual batch ID from the previous step
BATCH_ID = batch_id  # or paste it as a string like "batch_abc123"

batch_status = check_batch_status(client, BATCH_ID)


Batch ID: batch_68ea113f29ac8190810b220833f280ff
Status: validating
Total requests: 0
Completed: 0
Failed: 0


In [16]:
# Optional: Monitor batch progress with periodic checks
# This will check every 60 seconds until completion (or you can stop it manually)

def wait_for_batch_completion(client: OpenAI, batch_id: str, check_interval: int = 60):
    """
    Poll the batch status until it's completed or failed.
    
    Args:
        client: OpenAI client
        batch_id: The batch job ID
        check_interval: Seconds between status checks (default: 60)
    """
    print(f"Monitoring batch {batch_id}...")
    print("Press Ctrl+C to stop monitoring (batch will continue running)")
    
    try:
        while True:
            batch = client.batches.retrieve(batch_id)
            
            print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}]")
            print(f"Status: {batch.status}")
            print(f"Progress: {batch.request_counts.completed}/{batch.request_counts.total} completed")
            
            if batch.request_counts.failed > 0:
                print(f"Failed: {batch.request_counts.failed}")
            
            if batch.status in ["completed", "failed", "cancelled", "expired"]:
                print(f"\nBatch finished with status: {batch.status}")
                break
            
            print(f"Next check in {check_interval} seconds...")
            time.sleep(check_interval)
            
    except KeyboardInterrupt:
        print("\n\nMonitoring stopped. Batch is still running in the background.")
        print(f"Use check_batch_status(client, '{batch_id}') to check status later.")

# Uncomment to auto-monitor:
# wait_for_batch_completion(client, BATCH_ID, check_interval=60)

## Download and Process Results

Once the batch is completed, run this cell to download results and process them.

In [17]:
# Download results (only works when batch is completed)
download_batch_results(client, BATCH_ID, BATCH_OUTPUT_FILE)

# Parse the results
results, failures = parse_batch_results(BATCH_OUTPUT_FILE, valid_studies[:NUM_STUDIES])

# Show summary
tracker.summary()

print(f"\nSuccessfully processed: {len(results)} outcomes")
print(f"Failed: {len(failures)} outcomes")
if failures:
    print("\nFirst 5 failures:")
    for nct_id, error in failures[:5]:
        print(f"  {nct_id}: {error}")

Results saved to ../data/batch_output.jsonl
Total API calls: 3500
Total input tokens: 7,719,839
Total output tokens: 4,534,203
Estimated cost (Batch API with 50% discount): $5.4992
  (Standard API would cost: $10.9984)

Successfully processed: 3487 outcomes
Failed: 3849 outcomes

First 5 failures:
  NCT02026115: Parse error: 1 validation error for LLMResponse
  Value error, Question does not match the expected format. Got: Did tablet-based pain reporting with nurse decision support and tailored multimedia education improve analgesic adherence (percent of prescribed pain doses taken) in hospice patients given the tablet program with nurse decision support compared to usual hospice care with pain reports only after 1 week (days 1–7)?, Expected: Did tablet-based pain reporting with nurse decision support and tailored multimedia education improve analgesic adherence (percent of prescribed pain medicine doses taken) in hospice patients given the tablet program with nurse decision support co

In [18]:
import ujson as json
for result in results:
    with open(os.path.join(STUDIES_PROCESSED_OUTPUT_DIR, f"{result.study.nct_id}_{result.outcome_id}_llm_response.json"), "w") as f:
        json.dump(result.to_dict(), f, indent=4)

In [19]:
def get_num_participants(groups) -> int:
    n = 0
    for g in groups:
        num = g.num_participants
        if isinstance(num, int):
            n += num
        elif isinstance(num, str):
            new_num = ''
            for c in num:
                if c.isdigit():
                    new_num += c
            n += int(new_num)

    return n

def mk_card(r: ProcessingInformation) -> dict:
    """Create a card dictionary from the processing information."""
    o = next(o for o in r.study.primary_outcomes if o.id == r.outcome_id)
    p_value = f"{o.p_value.comparator}{o.p_value.value}"
    if p_value.startswith("="):
        p_value = p_value[1:]

    success = (o.p_value.value < 0.05 and o.p_value.comparator != ">") or p_value == "<0.05"

    return {
        "study": {
            "nct_id": r.study.nct_id,
            "title": r.study.title,
            "brief_description": r.study.brief_description,
        },
        "card_id": r.outcome_id,
        "front_details": {
            "question": r.llm_response.question,
            "intervention_fragment": r.llm_response.intervention_fragment,
            "intervention_group_fragment": r.llm_response.intervention_group_fragment,
            "outcome_fragment": r.llm_response.outcome_fragment,
            "comparator_group_fragment": r.llm_response.comparator_group_fragment,
            "timeframe_fragment": r.llm_response.timeframe_fragment,
        },
        "p_value": p_value,
        "num_participants": get_num_participants(o.groups),
        "success": success,
        "conditions": r.study.conditions,
        "keywords": r.study.keywords,
        "decks": r.study.decks,
        "understandability_score": r.llm_response.understandability_score,
    }

cards = []
n_fail = 0
for r in results:
    try:
        card = mk_card(r)
        cards.append(card)
    except Exception as e:
        n_fail += 1
        print(f"Failed to create card for {r.study.nct_id} outcome {r.outcome_id}: {str(e)}")
with open(os.path.join(CARDS_OUTPUT_DIR, "cards.json"), "w") as f:
    json.dump(cards, f, indent=4)

with open(os.path.join(CARDS_OUTPUT_DIR, "questions.txt"), "w") as f:
    for card in cards:
        f.write(card["front_details"]["question"] + "\n")

Failed to create card for NCT04108429 outcome NCT04108429_po_2: invalid literal for int() with base 10: ''
Failed to create card for NCT01925404 outcome NCT01925404_po_0: invalid literal for int() with base 10: ''
Failed to create card for NCT04682730 outcome NCT04682730_po_0: invalid literal for int() with base 10: ''


In [15]:
from pprint import pprint
for r in results:
    o = next(o for o in r.study.primary_outcomes if o.id == r.outcome_id)
    succ = o.p_value.value < 0.05 and o.p_value.comparator is not ">"
    print(f"Study {r.study.nct_id} {r.study.title}:")
    pprint(f"  Question: {r.llm_response.question}")
    print(f"  Answer: {'YES' if succ else 'NO'}")
    print('-'*60)
    
o

Study NCT02026115 Computerized PAINRelieveIt Protocol for Cancer Pain Control in Hospice:
('  Question: Did tablet-based pain reporting with nurse decision support and '
 'tailored multimedia education improve analgesic adherence (percent of '
 'prescribed pain meds taken) in home hospice cancer patients compared to '
 'usual hospice care with pain-report summary after 1 week?')
  Answer: NO
------------------------------------------------------------
Study NCT00802204 Dopamine and Insulin Resistance:
('  Question: Did a very low-calorie diet (VLCD) improve striatal dopamine D2 '
 'receptor binding (DRD2 binding potential) in obese participants after the '
 'diet compared to obese participants at baseline after 8–10 days?')
  Answer: NO
------------------------------------------------------------
Study NCT00802204 Dopamine and Insulin Resistance:
('  Question: Did a very-low-calorie diet (VLCD) improve striatal DRD2 '
 'receptor binding (binding potential) in obese participants after t

<>:4: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
/var/folders/sl/nv_r__ss6dj_bn0prgcrsgvr0000gn/T/ipykernel_10907/1349048745.py:4: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  succ = o.p_value.value < 0.05 and o.p_value.comparator is not ">"


PrimaryOutcome(nct_id='NCT02420990', id='NCT02420990_po_2', title='Treatment Attendance.', description='Treatment Attendance \\[sum of the total number of individual, family, and group sessions attended\\] and Medication Management Sessions \\[total number of sessions attended\\] were collected from agency records. Medication Use, coded as "1 = on" or "0 = off" medication at each follow-up point, was captured with the Services Assessment for Children and Adolescents', population_description='', timeframe='One Year', groups=[Group(id='OG000', title='Behavioral Only- Treatment', description='All participants will receive behavioral interventions (CASH-AA): family psycho-education in ADHD symptoms, executive functioning, and developmental impacts; family-based motivation and ADHD accommodation interventions; and academic training focused on home environment support and organizational skills.\n\nChanging Academic Support in the Home for Adolescents with ADHD (CASH-AA)', num_participants='5